In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import plotters


# Galaxy Image Classification — Numerical Results

**Single source of truth** for all computed quantities cited in the report. Every number in the report text must trace to a cell in this notebook.

In [ ]:
# Load all results
split_data = np.load("artifacts/split_indices.npz")
resnet_data = np.load("artifacts/resnet_result.npz")
custom_data = np.load("artifacts/custom_result.npz", allow_pickle=True)
sched_data = np.load("artifacts/scheduler_result.npz")
aug_data = np.load("artifacts/augmented_result.npz")
eval_data = np.load("artifacts/evaluation_results.npz")
label_data = np.load("artifacts/galaxy_zoo_labels.npz")

# Optional artifacts (NB 04 ablation, NB 05b class-weighted loss)
ablation_path = Path("artifacts/custom_cnn_ablation.npz")
cw_path = Path("artifacts/class_weighted_result.npz")
ablation_data = np.load(ablation_path, allow_pickle=True) if ablation_path.exists() else None
cw_data = np.load(cw_path) if cw_path.exists() else None


## Dataset Summary

In [ ]:
n_total = len(label_data["labels"])
n_train = len(split_data["train_idx"])
n_val = len(split_data["val_idx"])

dataset_summary = pd.DataFrame({
    "Quantity": ["Total galaxies", "Training set", "Validation set", "Number of labels"],
    "Value": [n_total, n_train, n_val, label_data["labels"].shape[1]],
})
dataset_summary

## Model Performance Summary

In [ ]:
rows = [
    ("Baseline (mean)",                  float(split_data["baseline_val_rmse"]), "N/A", 0),
    ("Custom CNN",                       float(custom_data["best_val_loss"]),    int(custom_data["best_epoch"]) + 1, int(custom_data["n_parameters"])),
    ("ResNet-18",                        float(resnet_data["best_val_loss"]),    int(resnet_data["best_epoch"]) + 1, int(resnet_data["n_parameters"])),
    ("ResNet-18 + LR Scheduler",         float(sched_data["best_val_loss"]),     int(sched_data["best_epoch"]) + 1,  int(sched_data["n_parameters"])),
    ("ResNet-18 + Aug + LR Scheduler",   float(aug_data["best_val_loss"]),       int(aug_data["best_epoch"]) + 1,    int(aug_data["n_parameters"])),
]
if cw_data is not None:
    rows.append(("ResNet-18 + Aug + LR + Tree-weighted loss",
                 float(cw_data["best_val_loss"]),
                 int(cw_data["best_epoch"]) + 1,
                 int(cw_data["n_parameters"])))

performance = pd.DataFrame(rows, columns=["Model", "Best Val RMSE", "Best Epoch", "Parameters"])
display(performance)

# Bar chart of model progression (mean / Custom / ResNet / +Sched / +Aug / [+CW]).
ax = plotters.plot_model_progression_bar(
    [r[0] for r in rows], [r[1] for r in rows],
)
plt.show()


## Per-Label Performance (Best Model)

In [ ]:
from ugdatalab.models.galaxy_zoo.constants import LABEL_COLUMNS, LABEL_DESCRIPTIVE

val_true = eval_data["val_true_labels"]
val_pred = eval_data["val_pred_labels"]

per_label_rmse = np.sqrt(((val_pred - val_true) ** 2).mean(axis=0))
per_label_bias = (val_pred - val_true).mean(axis=0)
per_label_scatter = (val_pred - val_true).std(axis=0)
label_desc_list = [LABEL_DESCRIPTIVE[c] for c in LABEL_COLUMNS]

per_label_df = pd.DataFrame({
    "Label": label_desc_list,
    "Bias": per_label_bias,
    "Scatter": per_label_scatter,
    "RMSE": per_label_rmse,
}).sort_values("RMSE", ascending=False).reset_index(drop=True)
display(per_label_df)

ax = plotters.plot_per_label_rmse_bar(label_desc_list, per_label_rmse)
plt.show()


## Merger Fraction

In [ ]:
merger_fraction = float(eval_data["merger_fraction"])
test_pred = eval_data["test_pred_labels"]
n_test = len(eval_data["test_galaxy_ids"])
merger_idx = LABEL_COLUMNS.index("Class8.6")
merger_probs = test_pred[:, merger_idx]

# Bootstrap a crude 68% interval on the mean-probability estimator.
rng = np.random.default_rng(42)
boot = np.array([
    merger_probs[rng.integers(0, n_test, size=n_test)].mean()
    for _ in range(1000)
])
ci_lo, ci_hi = np.percentile(boot, [16, 84])

threshold_rows = []
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    n_above = int(np.sum(merger_probs > thresh))
    threshold_rows.append({"threshold": thresh, "n_above": n_above,
                           "fraction (%)": 100 * n_above / n_test})
threshold_table = pd.DataFrame(threshold_rows)

print(f"Test set size: {n_test}")
print(f"Mean-probability merger fraction: {merger_fraction*100:.2f}%  "
      f"(68% bootstrap interval: {ci_lo*100:.2f}–{ci_hi*100:.2f}%)")
print()
print("Comparison to Lotz et al. 2011:")
print("  Observed major merger rate at z≈0:   ~0.01–0.03 Gyr^-1")
print("  Merger observability timescale:      ~0.5–1 Gyr")
print("  Predicted local merger fraction:     ~0.5–3 %")
print()
print("Threshold sweep on our estimate:")
display(threshold_table)
